In [ ]:
# 1. Setup: GPU check, Drive, data, tokenizer
import torch
from importlib.metadata import version, PackageNotFoundError

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU attached. Use Runtime > Change runtime type > T4 GPU, then run this cell again."
print("GPU:", torch.cuda.get_device_name(0))

for lib in ["torchao", "bitsandbytes", "peft", "transformers"]:
    try:
        print(lib, version(lib))
    except PackageNotFoundError:
        print(lib, "NOT installed")

from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
from transformers import AutoTokenizer

DATA_DIR = "/content/drive/MyDrive/promt shield"
MODEL_NAME = "Qwen/Qwen2.5-1.5B"
MAXLEN = 256

df = pd.read_csv(f"{DATA_DIR}/dataset_v2.csv", dtype=str, keep_default_na=False)
df = df[df["split"].isin(["train", "val"])].reset_index(drop=True)
assert set(df["split"]) == {"train", "val"}
df["y"] = (df["final_label"] == "attack").astype(int)
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
print("train rows:", len(train_df), "| val rows:", len(val_df))

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
print("pad token id:", tok.pad_token_id, "| padding side:", tok.padding_side)

torch: 2.14.1+cu130 | CUDA available: True
GPU: Tesla T4
torchao NOT installed
bitsandbytes 0.50.2
peft 0.21.2
transformers 5.22.1
Mounted at /content/drive
train rows: 17413 | val rows: 3298


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

pad token id: 151643 | padding side: right


In [ ]:
# 2. Imports and settings
import os
import gc
import time
import numpy as np
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

SEED = 42
BATCH, ACCUM, LR, EPOCHS, WARMUP = 8, 2, 2e-4, 2, 0.03
device = torch.device("cuda")
RESULTS_DIR = f"{DATA_DIR}/results"
CKPT_DIR = f"{DATA_DIR}/checkpoints"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print("GPU memory in use at the start (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

GPU memory in use at the start (GB): 0.0


In [ ]:
# 3. Dataset and batching
class PromptDataset(Dataset):
    def __init__(self, frame):
        texts = [t[:4000] for t in frame["text"]]
        self.input_ids = tok(texts, truncation=True, max_length=MAXLEN)["input_ids"]
        self.labels = frame["y"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {"input_ids": self.input_ids[i], "label": self.labels[i]}


def collate(items):
    seqs = [torch.tensor(x["input_ids"]) for x in items]
    lengths = torch.tensor([len(s) for s in seqs])
    input_ids = pad_sequence(seqs, batch_first=True, padding_value=tok.pad_token_id)
    attention_mask = (torch.arange(input_ids.shape[1])[None, :] < lengths[:, None]).long()
    labels = torch.tensor([x["label"] for x in items])
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [ ]:
# 4. Model builder: QLoRA (4-bit base model + LoRA adapters)
def lora_config():
    return LoraConfig(
        task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )


def build_model_4bit():
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, quantization_config=bnb_cfg, dtype=torch.float16, device_map={"": 0},
    )
    base.config.pad_token_id = tok.pad_token_id
    m = get_peft_model(base, lora_config())
    for q in m.parameters():
        if q.requires_grad:
            q.data = q.data.float()
    return m

In [ ]:
# 5. Prediction and metrics
def predict_probs(m, frame, batch_size=32):
    m.eval()
    loader = DataLoader(PromptDataset(frame), batch_size=batch_size, shuffle=False, collate_fn=collate)
    probs = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = m(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
            probs.extend(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().tolist())
    return np.array(probs)


def metrics(y, prob):
    pred = (prob >= 0.5).astype(int)
    cut = np.quantile(prob[y == 0], 0.99)
    return {
        "ap": average_precision_score(y, prob),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "recall_at_1pct_fp": float((prob[y == 1] > cut).mean()),
        "safe_flagged": int(((y == 0) & (pred == 1)).sum()),
    }

In [ ]:
# 6. Training and evaluation function
def train_and_eval(tag, build_fn):
    torch.manual_seed(SEED)
    model = build_fn()
    model.print_trainable_parameters()
    print("GPU memory after loading (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

    gen = torch.Generator()
    gen.manual_seed(SEED)
    loader = DataLoader(PromptDataset(train_df), batch_size=BATCH, shuffle=True,
                        drop_last=True, collate_fn=collate, generator=gen)
    trainable = [q for q in model.parameters() if q.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda")

    total_updates = EPOCHS * (len(loader) // ACCUM)
    warmup = int(WARMUP * total_updates)

    def lr_lambda(step):
        if step < warmup:
            return (step + 1) / warmup
        return max(0.0, (total_updates - step) / (total_updates - warmup))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    y_val = val_df["y"].values
    rows = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        torch.cuda.reset_peak_memory_stats()
        t_start = time.time()
        running, n_run = 0.0, 0
        for i, batch in enumerate(loader, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(**batch)
            scaler.scale(out.loss / ACCUM).backward()
            running += out.loss.item()
            n_run += 1
            if i % ACCUM == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                step = i // ACCUM
                if step % 100 == 0:
                    print(f"{tag} epoch {epoch} update {step}: avg loss {running / n_run:.3f} | {time.time() - t_start:.0f}s")
                    running, n_run = 0.0, 0
        torch.cuda.synchronize()
        train_seconds = time.time() - t_start
        peak_gb = torch.cuda.max_memory_allocated() / 1e9

        t_val = time.time()
        probs = predict_probs(model, val_df)
        torch.cuda.synchronize()
        val_seconds = time.time() - t_val
        m = metrics(y_val, probs)

        saved = val_df[["id", "source_dataset", "y"]].copy()
        saved["prob_attack"] = probs
        saved.to_csv(f"{RESULTS_DIR}/{tag}_val_probs_epoch{epoch}.csv", index=False)
        model.save_pretrained(f"{CKPT_DIR}/{tag}_epoch{epoch}")

        rows.append({
            "model": tag, "epoch": epoch,
            "train_minutes": train_seconds / 60, "train_peak_gb": peak_gb,
            "val_prompts_per_s": len(val_df) / val_seconds, **m,
        })
        pd.DataFrame(rows).round(4).to_csv(f"{RESULTS_DIR}/{tag}_epochs.csv", index=False)
        print(f"{tag} EPOCH {epoch} | val AP {m['ap']:.3f} | F1@0.5 {m['f1']:.3f} | "
              f"precision {m['precision']:.3f} recall {m['recall']:.3f} | recall@1%FP {m['recall_at_1pct_fp']:.3f} | "
              f"safe flagged {m['safe_flagged']} | train {train_seconds / 60:.1f} min | peak {peak_gb:.2f} GB")

    del model, optimizer, scaler, loader
    gc.collect()
    torch.cuda.empty_cache()
    return pd.DataFrame(rows)

In [ ]:
# 7. Run the QLoRA experiment
qlora_results = train_and_eval("qlora", build_model_4bit)
print()
print(qlora_results.round(3).to_string(index=False))

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 4,361,216 || all params: 1,548,078,592 || trainable%: 0.2817
GPU memory after loading (GB): 1.17


/tmp/ipykernel_2219/2133257114.py:45: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


qlora epoch 1 update 100: avg loss 0.647 | 178s
qlora epoch 1 update 200: avg loss 0.235 | 364s
qlora epoch 1 update 300: avg loss 0.192 | 546s
qlora epoch 1 update 400: avg loss 0.185 | 729s
qlora epoch 1 update 500: avg loss 0.140 | 911s
qlora epoch 1 update 600: avg loss 0.131 | 1092s
qlora epoch 1 update 700: avg loss 0.119 | 1275s
qlora epoch 1 update 800: avg loss 0.125 | 1459s
qlora epoch 1 update 900: avg loss 0.161 | 1642s
qlora epoch 1 update 1000: avg loss 0.106 | 1825s
qlora EPOCH 1 | val AP 0.972 | F1@0.5 0.917 | precision 0.938 recall 0.897 | recall@1%FP 0.849 | safe flagged 40 | train 33.1 min | peak 7.15 GB
qlora epoch 2 update 100: avg loss 0.051 | 180s
qlora epoch 2 update 200: avg loss 0.071 | 362s
qlora epoch 2 update 300: avg loss 0.081 | 543s
qlora epoch 2 update 400: avg loss 0.057 | 726s
qlora epoch 2 update 500: avg loss 0.060 | 908s
qlora epoch 2 update 600: avg loss 0.062 | 1088s
qlora epoch 2 update 700: avg loss 0.053 | 1271s
qlora epoch 2 update 800: avg l